In [1]:
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize) 

import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [2]:
default = load_data('default')
default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [3]:
design = MS(['income', 'balance'])

X = design.fit_transform(default)
y = default['default'] == 'Yes'

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=1)
lr = sm.GLM(y_train, 
            X_train,
            family=sm.families.Binomial())

results = lr.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,-11.858100,0.528000,-22.477,0.0
income,0.000022,0.000006,3.707,0.0
balance,0.005900,0.000000,21.078,0.0


In [4]:
results.bse

intercept    0.527555
income       0.000006
balance      0.000278
dtype: float64

In [5]:
# function that fits the model and returns the coefficients
def boot_fn(df: pd.DataFrame, idx: np.ndarray):
    design = MS(['income', 'balance'])

    data = df.loc[idx]
    
    X = design.fit_transform(data)
    y = data['default'] == 'Yes'
    
    lr = sm.GLM(y, 
                X,
                family=sm.families.Binomial())

    results = lr.fit()
    return results.params

In [6]:
# function that calculates the standard error
def boot_SE(func, D, n = None, B = 1000, seed = 0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index, n, replace = True)
        value = func(D, idx)
        first_ += value
        second_ += value ** 2
    return np.sqrt(second_ / B - (first_ / B) ** 2)

In [7]:
params_SE = boot_SE(boot_fn, default,  B = 1000, seed = 0)
params_SE

intercept    0.435692
income       0.000005
balance      0.000230
dtype: float64

In [8]:
print(results.bse)
print(params_SE)

intercept    0.527555
income       0.000006
balance      0.000278
dtype: float64
intercept    0.435692
income       0.000005
balance      0.000230
dtype: float64


In [9]:
# the standard errors are not too different 